# Multimodal product recommendation, trained from scratch (Colab GPU)

This notebook trains every model in the study **from random initialisation** (no pretrained weights) on the full
44k-product catalog, then produces the result tables and figures, plus a zip for the local app.

**Before you start**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Click **Runtime → Run all**.
3. When asked, **allow Google Drive access** (all progress is saved there), then **upload `mmrec_code.zip`**
   (it is in the project folder `colab/`).

**How long:** about 3 hours on a free T4 GPU. If Colab disconnects, just **Run all** again: finished runs are
saved in Drive and are skipped, so it continues where it stopped.

**At the end** a file `mmrec_scratch_results.zip` is downloaded. Unzip it into the project folder
`D:\D drive\Project\claude\multimodal_recsys` (allow it to merge folders), then start the app as usual.

In [ ]:
# 1. Check that a GPU is attached
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Run all again"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Google Drive: everything (data cache, runs, checkpoints) is kept here, so a disconnect loses nothing
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
OUT = pathlib.Path("/content/drive/MyDrive/mmrec_scratch")
OUT.mkdir(parents=True, exist_ok=True)
print("Saving to", OUT)

In [ ]:
# 3. Project code: uploaded once, then reused from Drive
import shutil, zipfile
code_zip = OUT / "mmrec_code.zip"
if not code_zip.exists():
    from google.colab import files
    print("Upload mmrec_code.zip (from the project folder colab/)")
    up = files.upload()
    name = next(iter(up))
    shutil.move(name, code_zip)
PROJ = pathlib.Path("/content/mmrec")
if PROJ.exists():
    shutil.rmtree(PROJ)
zipfile.ZipFile(code_zip).extractall(PROJ)
os.chdir(PROJ)
print(sorted(p.name for p in PROJ.iterdir()))

In [ ]:
# 4. Libraries (Colab already has torch, pandas, scikit-learn, matplotlib)
!pip -q install datasets

In [ ]:
# 5. Dataset: full 44k catalog from Hugging Face -> 64x64 images, tokenizer, 70/15/15 split (cached in Drive)
data_cache = OUT / "data_scratch"
local = PROJ / "data" / "scratch"
if (data_cache / "meta.csv").exists():
    shutil.copytree(data_cache, local, dirs_exist_ok=True)
    print("Loaded cached dataset from Drive")
else:
    !python -m src.scratch.data --source hf
    shutil.copytree(local, data_cache, dirs_exist_ok=True)
import pandas as pd
m = pd.read_csv(local / "meta.csv")
print(len(m), "products,", m.articleType.nunique(), "article types"); print(m.split.value_counts())

In [ ]:
# 6. The full study: optimizer tuning -> SGD/Adam/AdamW -> 6 methods -> ablations (about 3 hours, resumable)
!python -m src.scratch.experiments --out "{OUT}" --stages tune optimizers methods ablations

In [ ]:
# 7. Tables, significance tests, figures and the export for the local app
!python -m src.scratch.experiments --out "{OUT}" --stages report export --bundle /content/mmrec_scratch_results.zip
shutil.copy("/content/mmrec_scratch_results.zip", OUT / "mmrec_scratch_results.zip")

In [ ]:
# 8. Look at the results
from IPython.display import Image, display
res = OUT / "results" / "scratch"
print(pd.read_csv(res / "method_comparison.csv").round(3).to_string(index=False))
print(pd.read_csv(res / "optimizer_comparison.csv").round(3).to_string(index=False))
for f in ["method_comparison.png", "optimizer_curves.png", "optimizer_comparison.png", "ablations.png"]:
    display(Image(filename=str(res / f)))

In [ ]:
# 9. Download the results zip (also saved in Google Drive: MyDrive/mmrec_scratch/mmrec_scratch_results.zip)
from google.colab import files
files.download("/content/mmrec_scratch_results.zip")